In [ ]:
import os
import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from tqdm import tqdm
from timm import create_model  # ใช้ timm เพื่อโหลด HRNet-W48


In [ ]:
# 1. ตรวจสอบว่าสามารถใช้ GPU ได้หรือไม่
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# 2. กำหนดโฟลเดอร์สำหรับข้อมูล
data_dir = "/home/superairt02/Desktop/TermPaper2024/Rotation 2024/Classification model/Classification model"  # สมมติว่าคุณมีโฟลเดอร์จัดรูปแบบข้อมูล
train_dir = os.path.join(data_dir, "train")
val_dir = os.path.join(data_dir, "val")
test_dir = os.path.join(data_dir, "test")

In [ ]:
# 3. Transformations (ไม่มี augmentation)
transform = transforms.Compose([
    transforms.Resize((512, 512)),  # ใช้ขนาดภาพ 512x512
    transforms.ToTensor(),    
])


In [ ]:
batch_size = 2
# 4. DataLoader
datasets = {
    "train": datasets.ImageFolder(train_dir, transform=transform),
    "val": datasets.ImageFolder(val_dir, transform=transform),
    "test": datasets.ImageFolder(test_dir, transform=transform),
}

dataloaders = {
    "train": DataLoader(datasets["train"], batch_size=batch_size, shuffle=True),
    "val": DataLoader(datasets["val"], batch_size=batch_size, shuffle=False),
    "test": DataLoader(datasets["test"], batch_size=batch_size, shuffle=False),
}

In [ ]:
# โมเดล HRNet_w48
model = create_model("hrnet_w48", pretrained=True, num_classes=3)
model = model.to(device)

In [ ]:
# 6. Loss และ Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [ ]:
# Train
def train_model(model, criterion, optimizer, dataloaders, num_epochs=200):
    best_model_wts = model.state_dict()
    best_acc = 0.0

    # สำหรับเก็บค่า loss และ accuracy
    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

    # ตรวจสอบจำนวนตัวอย่างใน dataloaders
    for phase in ["train", "val"]:
        print(f"Number of samples in {phase} dataset: {len(dataloaders[phase].dataset)}")
        if len(dataloaders[phase].dataset) == 0:
            print(f"Warning: {phase} dataset is empty!")

    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        print("-" * 10)

        for phase in ["train", "val"]:
            if phase == "train":
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            # ใช้ tqdm เพื่อแสดง Progress Bar
            with tqdm(total=len(dataloaders[phase]), desc=f"{phase} Progress", leave=False) as pbar:
                for inputs, labels in dataloaders[phase]:
                    inputs, labels = inputs.to(device), labels.to(device)

                    optimizer.zero_grad()

                    with torch.set_grad_enabled(phase == "train"):
                        outputs = model(inputs)
                        _, preds = torch.max(outputs, 1)
                        loss = criterion(outputs, labels)

                        if phase == "train":
                            loss.backward()
                            optimizer.step()

                    running_loss += loss.item() * inputs.size(0)
                    running_corrects += torch.sum(preds == labels.data)

                    # อัปเดต Progress Bar
                    pbar.update(1)

            epoch_loss = running_loss / len(dataloaders[phase].dataset)
            epoch_acc = running_corrects.double() / len(dataloaders[phase].dataset)

            print(f"{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}")

            # เก็บค่า loss และ accuracy ไว้ใน history
            if phase == "train":
                history["train_loss"].append(epoch_loss)
                history["train_acc"].append(epoch_acc.item())
            else:
                history["val_loss"].append(epoch_loss)
                history["val_acc"].append(epoch_acc.item())

            # Save the best model weights
            if phase == "val" and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = model.state_dict()
                torch.save(best_model_wts, "Best_classification3.pth")  # บันทึกโมเดลที่ดีที่สุด

    print("Training complete")
    print(f"Best val Acc: {best_acc:.4f}")

    # Load best model weights into the model
    model.load_state_dict(best_model_wts)
    return model, history


In [ ]:
# 8. Train โมเดล
num_epochs = 200
model, history = train_model(model, criterion, optimizer, dataloaders, num_epochs=num_epochs)

In [ ]:
# พล็อตกราฟ Loss และ Accuracy พร้อมแสดงจุด Validation Accuracy ที่ดีที่สุด
def plot_training_history(history):
    # ดึงข้อมูลจาก history
    epochs = range(1, len(history["train_loss"]) + 1)
    train_losses = history["train_loss"]
    val_losses = history["val_loss"]
    train_accuracies = history["train_acc"]
    val_accuracies = history["val_acc"]

    # หาค่า Accuracy ที่ดีที่สุดและ epoch ที่เกี่ยวข้อง
    best_val_acc = max(val_accuracies)
    best_epoch = val_accuracies.index(best_val_acc) + 1

    # วาดกราฟ Training และ Validation Loss
    plt.figure(figsize=(12, 6))

    # กราฟ Loss
    plt.subplot(1, 2, 1)
    plt.plot(epochs, train_losses, label='Training Loss', marker='o')
    plt.plot(epochs, val_losses, label='Validation Loss', marker='o')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)

    # กราฟ Accuracy
    plt.subplot(1, 2, 2)
    plt.plot(epochs, train_accuracies, label='Training Accuracy', marker='o')
    plt.plot(epochs, val_accuracies, label='Validation Accuracy', marker='o')
    plt.scatter(best_epoch, best_val_acc, color='red', label=f'Best Val Acc: {best_val_acc:.4f} (Epoch {best_epoch})')
    plt.title('Training and Validation Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)

    plt.tight_layout()
    plt.show()

plot_training_history(history)


In [ ]:
import os
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd  # สำหรับบันทึกข้อมูลเป็น Excel
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report

# โหลดโมเดลที่ดีที่สุด
best_model_path = "Best_classification3.pth"
if os.path.exists(best_model_path):
    model.load_state_dict(torch.load(best_model_path))
    print(f"Loaded best model from {best_model_path}")
else:
    print("Best model file not found. Using the current model.")

# ฟังก์ชันประเมินผลโมเดล (เพิ่มการคำนวณ probability ด้วย)
def evaluate_model(model, dataloader):
    model.eval()
    all_preds = []
    all_labels = []
    all_images = []
    all_probs = []  # เก็บ probability ของการทำนายแต่ละภาพ

    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            # คำนวณ probability จาก logits ด้วย softmax
            probs = torch.softmax(outputs, dim=1)
            # ดึงค่า probability ที่มากที่สุด (สำหรับคลาสที่ทำนาย)
            max_probs, preds = torch.max(probs, 1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_images.extend(inputs.cpu().numpy())
            all_probs.extend(max_probs.cpu().numpy())

    return np.array(all_labels), np.array(all_preds), np.array(all_images), np.array(all_probs)

# ประเมินผลบน Test Dataset
labels, preds, images, probs = evaluate_model(model, dataloaders["test"])

# ดึงชื่อไฟล์รูปภาพจาก test dataset (สมมุติว่าใช้ ImageFolder)
# หมายเหตุ: ต้องแน่ใจว่า test dataloader ตั้งค่า shuffle=False เพื่อให้ลำดับตรงกัน
image_names = [os.path.basename(path) for (path, _) in dataloaders["test"].dataset.samples]

# บันทึกข้อมูลชื่อรูปและ probability ลงในไฟล์ Excel
df = pd.DataFrame({'Image Name': image_names, 'Probability': probs})
df.to_excel('prediction_probabilities3.xlsx', index=False)
print("Saved prediction probabilities to prediction_probabilities.xlsx")

# คำนวณ Confusion Matrix
conf_matrix = confusion_matrix(labels, preds)
class_names = datasets["test"].classes

# ฟังก์ชันแสดง Confusion Matrix
def plot_confusion_matrix(cm, class_names):
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
    plt.xlabel("Predicted Labels")
    plt.ylabel("Actual Labels")
    plt.title("Confusion Matrix")
    plt.show()

plot_confusion_matrix(conf_matrix, class_names)

# คำนวณค่ามาตรการต่าง ๆ
accuracy = accuracy_score(labels, preds)
class_report = classification_report(labels, preds, target_names=class_names, output_dict=True)

sensitivity = {}
specificity = {}
accuracy_per_class = {}

for i, class_name in enumerate(class_names):
    tp = conf_matrix[i, i]
    fn = conf_matrix[i, :].sum() - tp
    fp = conf_matrix[:, i].sum() - tp
    tn = conf_matrix.sum() - (tp + fn + fp)

    sensitivity[class_name] = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity[class_name] = tn / (tn + fp) if (tn + fp) > 0 else 0

    # คำนวณ Accuracy แยกตาม Class
    class_labels = (np.array(labels) == i)
    class_preds = (np.array(preds) == i)
    correct = np.sum(class_labels & class_preds)
    total = np.sum(class_labels)
    accuracy_per_class[class_name] = correct / total if total > 0 else 0

# แสดงผลลัพธ์ของแต่ละ Class
print(f"Overall Accuracy: {accuracy:.4f}")
for class_name in class_names:
    print(f"\nClass: {class_name}")
    print(f"  Sensitivity: {sensitivity[class_name]:.4f}")
    print(f"  Specificity: {specificity[class_name]:.4f}")
    print(f"  Accuracy: {accuracy_per_class[class_name]:.4f}")
    print(f"  F1-score: {class_report[class_name]['f1-score']:.4f}")

# คำนวณค่าเฉลี่ยและส่วนเบี่ยงเบนมาตรฐาน
def calculate_mean_sd(values):
    mean = np.mean(values)
    sd = np.std(values)
    return mean, sd

sensitivity_values = list(sensitivity.values())
specificity_values = list(specificity.values())
accuracy_values = list(accuracy_per_class.values())
f1_scores = [class_report[class_name]["f1-score"] for class_name in class_names]

mean_sensitivity, sd_sensitivity = calculate_mean_sd(sensitivity_values)
mean_specificity, sd_specificity = calculate_mean_sd(specificity_values)
mean_accuracy, sd_accuracy = calculate_mean_sd(accuracy_values)
mean_f1_score, sd_f1_score = calculate_mean_sd(f1_scores)

# แสดงค่าเฉลี่ยและส่วนเบี่ยงเบนมาตรฐาน
print("\nSummary (Mean ± SD):")
print(f"Sensitivity: {mean_sensitivity:.4f} ± {sd_sensitivity:.4f}")
print(f"Specificity: {mean_specificity:.4f} ± {sd_specificity:.4f}")
print(f"Accuracy: {mean_accuracy:.4f} ± {sd_accuracy:.4f}")
print(f"F1-score: {mean_f1_score:.4f} ± {sd_f1_score:.4f}")

# ฟังก์ชันแสดงภาพผลการทำนาย (ปรับปรุงเพื่อแสดงชื่อรูปและ probability ด้วย)
def visualize_predictions_batch(images, labels, preds, probs, image_names, class_names, batch_size=10):
    num_batches = len(images) // batch_size + (1 if len(images) % batch_size != 0 else 0)

    for batch_idx in range(num_batches):
        start_idx = batch_idx * batch_size
        end_idx = min((batch_idx + 1) * batch_size, len(images))

        plt.figure(figsize=(20, 10))
        for i, idx in enumerate(range(start_idx, end_idx)):
            image = images[idx].transpose(1, 2, 0)  # เปลี่ยนรูปแบบจาก CHW เป็น HWC
            true_label = class_names[labels[idx]]
            pred_label = class_names[preds[idx]]
            prob = probs[idx]
            img_name = image_names[idx]
            
            plt.subplot(4, 5, i + 1)
            plt.imshow(image)
            plt.axis("off")
            plt.title(f"Name: {img_name}\nTrue: {true_label}\nPred: {pred_label}\nProb: {prob:.2f}",
                      color="green" if true_label == pred_label else "red")

        plt.tight_layout()
        plt.show()

# เรียกใช้งานฟังก์ชันแสดงภาพ (รวมชื่อรูปและ probability ด้วย)
visualize_predictions_batch(images, labels, preds, probs, image_names, class_names, batch_size=10)